# mCount benchmark

Runs the mycol fine-tuned Cellpose model on every well of all ten mCount plates and scores it against
MCount and NICE on the same images.

- **Writes** - `../assets/cellpose_count_evaluation.csv`, the scoring Figure 2 panel **g** plots
- **Unit** - one whole well image (`image_X__<well>.jpg`, 96 per plate). Ground truth is the sum of
  `counted` over that well's segment rows in `groundtruth.csv`; MCount's count is the sum of its
  per-segment predictions (`d_0.5_lambda_38.csv`, its best grid point here); NICE's comes from `NICE.csv`
- **Excluded** - any well holding a `counted == -1` segment (the mCount authors' "uncountable" flag,
  so the well has no reliable total), any well with a true count of 0 (percentage error undefined),
  and the 13 wells the model was fine-tuned on. The summary cell prints the metric with and without
  those 13
- **Inference matches the app** - mycol resizes every upload to 512x512 in `add_image()`, and the
  model was fine-tuned on those 512x512 wells, so it is scored the same way. The resize, preprocess
  and mask helpers are imported from `src/`, not reimplemented
- **Needs** - `mycol_saved_session_mcount50.zip` in `case_study_1/mcount/`; export it from the app's
  Downloads page if missing
- **Kernel** - `mycol_colonies_env`, run top to bottom


In [ ]:
# --- locate the mycol repo root (so the app helpers import) and set paths ---
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    """Walk up from `start` until the mycol source tree is found."""
    for p in [start, *start.parents]:
        if (p / "src" / "helpers" / "cellpose_functions.py").exists():
            return p
    raise RuntimeError("Could not locate the mycol repo root above %s" % start)


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

MCOUNT_DATA = REPO_ROOT / "case_study_1" / "mcount"
HERE = Path("output")               
HERE.mkdir(exist_ok=True)
FIGURE_ASSETS = Path("..") / "assets"   
MODEL_ZIP = MCOUNT_DATA / "mycol_saved_session_mcount50.zip"
RESULTS_DIR = MCOUNT_DATA / "mcount_results"
OUTPUT_CSV = FIGURE_ASSETS / "cellpose_count_evaluation.csv"

print("repo root :", REPO_ROOT)
print("model zip :", MODEL_ZIP, "(exists:", MODEL_ZIP.exists(), ")")
print("results   :", RESULTS_DIR, "(exists:", RESULTS_DIR.exists(), ")")


In [ ]:
# --- the fine-tuned Cellpose model and the inference settings the app recorded for it ---
import tempfile
import zipfile

import pandas as pd

extract_dir = Path(tempfile.mkdtemp(prefix="mcount_model_"))
with zipfile.ZipFile(MODEL_ZIP) as z:
    z.extractall(extract_dir)

pt_files = list(extract_dir.rglob("cellpose_model.pt")) or list(extract_dir.rglob("*.pt"))
if not pt_files:
    raise FileNotFoundError(
        f"No Cellpose model (.pt) inside {MODEL_ZIP.name}. Export the fine-tuned model "
        "from the mycol Downloads page, place it here, and re-run this cell.")
MODEL_PATH = pt_files[0]

# mycol stores the exact inference settings alongside the model, so the counts here match
# the app's. The app writes 0 for "estimate the diameter automatically"; Cellpose wants None.
hp = pd.read_csv(next(extract_dir.rglob("cellpose_inference_hyperparameters.csv")))
stored = dict(zip(hp["parameter"].astype(str), hp["value"].astype(float)))
INFER = {
    "diameter": stored["diameter"] or None,
    "cellprob_threshold": stored["cellprob_threshold"],
    "flow_threshold": stored["flow_threshold"],
    "min_size": int(stored["min_size"]),
    "niter": int(stored["niter"]),
}

print("model:", MODEL_PATH.name)
print("inference settings:", INFER)


In [ ]:
# --- load the model ---
import torch
from cellpose import models as cp_models

use_gpu = torch.cuda.is_available() or torch.backends.mps.is_available()
model = cp_models.CellposeModel(pretrained_model=str(MODEL_PATH), gpu=use_gpu)
print("Cellpose model loaded (gpu=%s)" % use_gpu)


In [ ]:
import numpy as np
from PIL import Image

from src.helpers.cellpose_functions import (
    preprocess_for_cellpose,
    convert_cellpose_mask_to_single_array,
)
from src.helpers.densenet_functions import resize_with_aspect_ratio

UPLOAD_SIZE = 512   # app default: ss["resize_on_upload"] is True, add_image() -> 512


def app_record(image_path: Path) -> dict:
    """Build the record the app would hold after add_image(), i.e. the 512x512 working image."""
    raw = np.array(Image.open(image_path).convert("RGB"), dtype=np.uint8)   # as add_image() reads it
    img = resize_with_aspect_ratio(raw, UPLOAD_SIZE)                        # the upload resize
    H, W = img.shape[:2]
    return {"image": img, "H": H, "W": W, "orig_H": raw.shape[0], "orig_W": raw.shape[1]}


def count_cells(image_path: Path) -> int:
    """Segment one image the way the app does and return the number of detected cells."""
    rec = app_record(image_path)
    im_in = preprocess_for_cellpose(rec)   # grayscale; Cellpose normalises internally
    masks_out, _, _ = model.eval(
        [im_in],
        channels=[0, 0],
        diameter=INFER["diameter"],
        cellprob_threshold=INFER["cellprob_threshold"],
        flow_threshold=INFER["flow_threshold"],
        min_size=INFER["min_size"],
        niter=INFER["niter"],
    )
    mask = masks_out[0] if isinstance(masks_out, (list, tuple)) else masks_out
    # segment_with_cellpose() writes the mask back at rec["H"], rec["W"] - the 512x512 working size
    labels = convert_cellpose_mask_to_single_array(mask, rec["H"], rec["W"])
    return int(np.unique(labels[labels > 0]).size)


In [ ]:
# --- ground truth + tool counts: per-well totals over each well's segments ---
import re

WELL_STEM = re.compile(r"^image_\d+__[A-H]\d+$")  # matches a well image stem, not a segment

MCOUNT_GRID = "d_0.5_lambda_38.csv"  # MCount's per-segment predictions (its best grid point on this data)


def true_counts_for_plate(plate_dir: Path) -> dict[str, int]:
    gt = pd.read_csv(plate_dir / "groundtruth.csv")
    # A `counted` of -1 flags a segment the mCount authors deemed uncountable
    gt = gt[gt["counted"] >= 0]
    # image_X__A1__7.jpg -> image_X__A1  (drop the __N.jpg segment suffix)
    well = gt["filename_seg"].str.replace(r"__\d+\.jpg$", "", regex=True)
    return gt.assign(well=well).groupby("well")["counted"].sum().astype(int).to_dict()


def wells_with_invalid_segment(plate_dir: Path) -> set:
    """Well stems (image_X__<well>) that contain at least one -1 (uncountable) segment."""
    gt = pd.read_csv(plate_dir / "groundtruth.csv")
    inv = gt[gt["counted"] < 0]
    return set(inv["filename_seg"].str.replace(r"__\d+\.jpg$", "", regex=True))


def mcount_counts_for_plate(plate_dir: Path, grid: str = MCOUNT_GRID) -> dict[str, int]:
    """MCount's whole-image count per well = sum of its per-segment predictions from the grid file."""
    mc = pd.read_csv(plate_dir / grid)
    well = mc["filename_seg"].str.replace(r"__\d+\.jpg$", "", regex=True)
    return mc.assign(well=well).groupby("well")["counted"].sum().round().astype(int).to_dict()


def nice_counts_for_plate(plate_dir: Path) -> dict[str, int]:
    """NICE tool's whole-image counts (from NICE.csv) - a published baseline the MCount paper compares
    against. Already one count per well (image_X__<well>.jpg), so no aggregation is needed."""
    nc = pd.read_csv(plate_dir / "NICE.csv")
    well = nc["filename_seg"].str.replace(r"\.jpg$", "", regex=True)
    return dict(zip(well, nc["counted"].astype(int)))


In [ ]:
# --- run the model over every well image in every plate ---
plate_dirs = sorted(
    (p for p in RESULTS_DIR.iterdir() if p.is_dir()),
    key=lambda p: int(re.search(r"\d+", p.name).group()),
)

def _well_sort_key(p: Path):
    well = p.stem.split("__")[-1]      # e.g. "A1", "H12"
    return (well[0], int(well[1:]))     # row letter, then column number


rows = []
for plate_dir in plate_dirs:
    truth = true_counts_for_plate(plate_dir)
    invalid = wells_with_invalid_segment(plate_dir)   # images to exclude (contain a -1 segment)
    mcount = mcount_counts_for_plate(plate_dir)        # MCount whole-image count per well
    nice = nice_counts_for_plate(plate_dir)            # NICE whole-image count per well
    well_imgs = sorted(
        (p for p in plate_dir.glob("*.jpg") if WELL_STEM.match(p.stem)),
        key=_well_sort_key,
    )
    for img_path in well_imgs:
        stem = img_path.stem
        rows.append(
            {
                "plate": plate_dir.name,
                "well": stem.split("__")[-1],
                "image": img_path.name,
                "model_count": count_cells(img_path),
                "mcount_count": mcount.get(stem),
                "nice_count": nice.get(stem),
                "true_count": truth.get(stem),
                "has_invalid_segment": stem in invalid,
            }
        )
    print(f"{plate_dir.name}: counted {len(well_imgs)} well images")

df = pd.DataFrame(rows)
print("\ntotal images counted:", len(df))
df.head()


In [ ]:
# --- per-image percentage error for every tool (images with a -1 segment excluded) + save CSV ---
df["true_count"] = df["true_count"].astype("Int64")
for col in ("mcount_count", "nice_count"):
    df[col] = df[col].astype("Int64")

# compare total image counts only where the ground truth is trustworthy:
#   - exclude images containing any -1 (uncountable) segment  -> true total unknowable
#   - exclude images whose true count is 0                    -> percentage error undefined
valid = ((df["true_count"] > 0) & ~df["has_invalid_segment"]).fillna(False)

# error for each tool on the same images (mycol Cellpose = model_count)
for prefix, col in [("", "model_count"), ("mcount_", "mcount_count"), ("nice_", "nice_count")]:
    d = df[col] - df["true_count"]
    df[f"{prefix}abs_pct_error"] = np.where(valid, (d.abs() / df["true_count"]) * 100, np.nan)
    df[f"{prefix}signed_pct_error"] = np.where(valid, (d / df["true_count"]) * 100, np.nan)

# Wells the model was fine-tuned on: they are not held-out, so the headline metric excludes them.
with zipfile.ZipFile(MODEL_ZIP) as _z:
    TRAIN_WELLS = {
        Path(n).name[: -len("_masks.tif")]
        for n in _z.namelist()
        if n.startswith("masks/") and n.endswith("_masks.tif")
    }
df["is_training_well"] = df["image"].str.replace(r"\.jpg$", "", regex=True).isin(TRAIN_WELLS)

df.to_csv(OUTPUT_CSV, index=False)
print("wrote", OUTPUT_CSV)
df.head()


In [ ]:
# --- mean percentage error across images: mycol Cellpose vs MCount vs NICE ---
# Two nested sets, both whole-image and both with -1 wells excluded:
#   comparable  = every scorable well (no -1 segment, true count > 0)
#   HELD OUT    = comparable minus the wells the model was fine-tuned on  <- the headline number
n_valid = int(valid.sum())
n_invalid_excl = int(df["has_invalid_segment"].sum())
n_zero_excl = int(((df["true_count"] == 0) & ~df["has_invalid_segment"]).sum())
held_out = valid & ~df["is_training_well"]
n_train_in = int((valid & df["is_training_well"]).sum())

print(f"wells total          : {len(df)}")
print(f"  excluded, -1 segment : {n_invalid_excl}")
print(f"  excluded, true count 0: {n_zero_excl}")
print(f"comparable wells     : {n_valid}")
print(f"  of which fine-tuned on: {n_train_in}  -> removed from the headline metric")
print(f"HELD-OUT wells       : {int(held_out.sum())}\n")


def _tool_errors(mask):
    return {
        "Cellpose (mycol)": [df.loc[mask, "abs_pct_error"].mean(), df.loc[mask, "signed_pct_error"].mean()],
        "MCount": [df.loc[mask, "mcount_abs_pct_error"].mean(), df.loc[mask, "mcount_signed_pct_error"].mean()],
        "NICE": [df.loc[mask, "nice_abs_pct_error"].mean(), df.loc[mask, "nice_signed_pct_error"].mean()],
    }


idx = ["Mean Absolute % Error", "Mean % Error (signed)"]
headline = pd.DataFrame(_tool_errors(held_out), index=idx).round(2)
all_comparable = pd.DataFrame(_tool_errors(valid), index=idx).round(2)

print(f"HEADLINE - held-out wells only (n = {int(held_out.sum())})   <- quote these in the paper")
print(headline.to_string())
print(f"\nFor reference - all comparable wells (n = {n_valid}, includes {n_train_in} fine-tuning wells)")
print(all_comparable.to_string())
# signed: positive = over-counts. MCount = sum of its per-segment predictions; NICE = from NICE.csv.
headline


In [ ]:
# --- optional: per-plate breakdown (held-out wells only), all tools ---
per_plate = (
    df[held_out]
    .groupby("plate")
    .agg(
        images=("image", "size"),
        true_total=("true_count", "sum"),
        cellpose_total=("model_count", "sum"),
        mcount_total=("mcount_count", "sum"),
        nice_total=("nice_count", "sum"),
        cellpose_mape=("abs_pct_error", "mean"),
        mcount_mape=("mcount_abs_pct_error", "mean"),
        nice_mape=("nice_abs_pct_error", "mean"),
    )
    .reindex([p.name for p in plate_dirs])
    .round(2)
)
per_plate


In [ ]:
# --- scatter: whole-image true vs predicted colony count, all three tools ---
import matplotlib.pyplot as plt

# Okabe-Ito CVD-safe: blue = Cellpose (mycol), vermillion = MCount, green = NICE
TVP = {"cellpose": "#0072B2", "mcount": "#D55E00", "nice": "#009E73"}
compared = df[held_out]                              # held-out wells: no -1 segment, true count > 0, not fine-tuned on


def _true_vs_pred(ax, true, pred, color, title, s, alpha, edge):
    true = np.asarray(pd.Series(true).astype("float64"))
    pred = np.asarray(pd.Series(pred).astype("float64"))
    lim = max(true.max(), pred.max()) * 1.05
    ax.plot([0, lim], [0, lim], color="#8a8a8a", lw=1.2, ls="--", zorder=1)          # y = x (perfect)
    ax.scatter(true, pred, s=s, alpha=alpha, color=color, zorder=2,
               edgecolors="white" if edge else "none", linewidths=0.5 if edge else 0)
    ax.set_xlim(-lim * 0.02, lim); ax.set_ylim(-lim * 0.02, lim); ax.set_aspect("equal")
    mape = (np.abs(pred - true) / true * 100).mean()
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.text(0.05, 0.95, f"MAPE {mape:.2f}%\nn = {len(true):,}", transform=ax.transAxes, va="top",
            ha="left", fontsize=9, bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="#dddddd"))
    ax.text(lim * 0.97, lim * 0.9, "y = x", color="#8a8a8a", fontsize=8, rotation=45,
            rotation_mode="anchor", va="bottom", ha="right")
    ax.set_xlabel("true count"); ax.set_ylabel("predicted count")
    ax.grid(True, lw=0.4, color="#ededed"); ax.set_axisbelow(True)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)


fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.7))
_true_vs_pred(axes[0], compared["true_count"], compared["model_count"], TVP["cellpose"], "mycol Cellpose", 28, 0.55, True)
_true_vs_pred(axes[1], compared["true_count"], compared["mcount_count"], TVP["mcount"], "MCount", 28, 0.55, True)
_true_vs_pred(axes[2], compared["true_count"], compared["nice_count"], TVP["nice"], "NICE", 28, 0.55, True)
fig.suptitle(f"Whole-image colony count: true vs predicted  ({len(compared)} held-out wells, no -1 segments)",
             fontsize=12.5, fontweight="bold", y=1.0)
fig.tight_layout()
fig.savefig(HERE / "true_vs_pred_images.png", dpi=150, bbox_inches="tight", facecolor="white")
plt.show()
